# Tensori și autograd

PyTorch combină un sistem de tensori accelerabil cu diferențiere automată. În această lecție tratăm formele tensorilor ca parte din specificația modelului și inspectăm explicit graful folosit la backpropagation.

## 1. Tensorul: date, formă și tip

Un tensor are valori, `shape`, `dtype` și `device`. Pentru un batch de imagini, convenția uzuală este `(N, C, H, W)`. Operațiile sunt valide numai când dimensiunile sunt compatibile sau pot fi extinse prin broadcasting.

In [1]:
import torch
torch.manual_seed(7)
X = torch.randn(4, 3)
W = torch.randn(2, 3)
b = torch.zeros(2)
logits = X @ W.T + b
print('X:', X.shape)
print('logits:', logits.shape)

X: torch.Size([4, 3])
logits: torch.Size([4, 2])


## 2. Broadcasting fără surprize

În $XW^T+b$, tensorul `b` cu forma `(2,)` este văzut ca `(1, 2)` și extins peste cele patru exemple. Broadcasting nu copiază neapărat memoria, dar o aliniere greșită poate produce un rezultat valid dimensional și incorect semantic. Notați formele înaintea operațiilor importante.

In [2]:
images = torch.arange(24).reshape(2, 3, 2, 2)
flattened = images.flatten(start_dim=1)
flattened.shape

torch.Size([2, 12])

## 3. Graful computațional

Dacă `requires_grad=True`, PyTorch înregistrează operațiile. Pentru $L(w)=(wx-y)^2$, regula lanțului dă

$$\frac{dL}{dw}=2(wx-y)x. $$

Apelul `backward()` propagă un cotangent de la scalarul final către frunze și acumulează rezultatul în `.grad`.

In [3]:
w = torch.tensor(1.0, requires_grad=True)
x, y = torch.tensor(3.0), torch.tensor(5.0)
loss = (w*x-y)**2
loss.backward()
print('autograd:', w.grad.item())
print('analitic:', (2*(w.detach()*x-y)*x).item())

autograd: -12.0
analitic: -12.0


## 4. Acumularea gradientelor

PyTorch adună gradientele la fiecare `backward()`. Într-o buclă de antrenare apelăm `optimizer.zero_grad()` înainte de următorul pas. Acumularea deliberată este însă utilă când simulăm un batch mare din mai multe micro-batch-uri.

Operațiile din `torch.no_grad()` nu intră în graf și sunt potrivite pentru evaluare sau actualizări manuale.

In [4]:
v = torch.tensor([1., 2., 3.], requires_grad=True)
z = v**2
z.backward(torch.tensor([1., 3., 2.]))  # produs vector–Jacobian
v.grad

tensor([ 2., 12., 12.])

## Laborator

1. Rescrie stratul liniar pentru un tensor `(batch, timp, caracteristici)`.
2. Verifică gradientul lui `torch.sigmoid` cu diferențe finite.
3. Provoacă și explică o eroare de operație in-place pe o frunză cu `requires_grad=True`.
4. Folosește `torch.autograd.grad` pentru a calcula o derivată de ordinul al doilea.